In [177]:
import pandas as pd

DATA_PATH = "../data/raw/E0.csv"

df_raw = pd.read_csv(DATA_PATH, parse_dates=["Date"],
    dayfirst=True)
df_raw.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,2025-08-15,20:00,Liverpool,Bournemouth,4,2,H,1,0,...,2.03,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86
1,E0,2025-08-16,12:30,Aston Villa,Newcastle,0,0,D,0,0,...,2.05,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86
2,E0,2025-08-16,15:00,Brighton,Fulham,1,1,D,0,0,...,1.83,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08
3,E0,2025-08-16,15:00,Sunderland,West Ham,3,0,H,0,0,...,1.95,1.90,1.97,1.95,1.95,1.94,1.86,1.78,2.02,1.97
4,E0,2025-08-16,15:00,Tottenham,Burnley,3,0,H,1,0,...,1.98,1.88,1.99,1.93,1.98,1.91,1.88,1.83,2.07,1.92


In [178]:
cols = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]

df = df_raw[cols].copy()
df.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,2025-08-15,Liverpool,Bournemouth,4,2,H
1,2025-08-16,Aston Villa,Newcastle,0,0,D
2,2025-08-16,Brighton,Fulham,1,1,D
3,2025-08-16,Sunderland,West Ham,3,0,H
4,2025-08-16,Tottenham,Burnley,3,0,H


In [179]:
df = df.rename(columns={
    "Date":"date",
    "HomeTeam":"home_team",
    "AwayTeam":"away_team",
    "FTHG":"home_goals",
    "FTAG":"away_goals",
    "FTR":"result"
})

df.head()

,date,home_team,away_team,home_goals,away_goals,result
0,2025-08-15,Liverpool,Bournemouth,4,2,H
1,2025-08-16,Aston Villa,Newcastle,0,0,D
2,2025-08-16,Brighton,Fulham,1,1,D
3,2025-08-16,Sunderland,West Ham,3,0,H
4,2025-08-16,Tottenham,Burnley,3,0,H


In [180]:
df["season"]="2024/25"

df.head()

,date,home_team,away_team,home_goals,away_goals,result,season
0,2025-08-15,Liverpool,Bournemouth,4,2,H,2024/25
1,2025-08-16,Aston Villa,Newcastle,0,0,D,2024/25
2,2025-08-16,Brighton,Fulham,1,1,D,2024/25
3,2025-08-16,Sunderland,West Ham,3,0,H,2024/25
4,2025-08-16,Tottenham,Burnley,3,0,H,2024/25


In [181]:
df["home_points"] = df["result"].apply(lambda x:3 if x=="H" else 1 if x=="D" else 0)
df["away_points"] = df["result"].apply(lambda x:3 if x=="A" else 1 if x=="D" else 0)

df.head()

,date,home_team,away_team,home_goals,away_goals,result,season,home_points,away_points
0,2025-08-15,Liverpool,Bournemouth,4,2,H,2024/25,3,0
1,2025-08-16,Aston Villa,Newcastle,0,0,D,2024/25,1,1
2,2025-08-16,Brighton,Fulham,1,1,D,2024/25,1,1
3,2025-08-16,Sunderland,West Ham,3,0,H,2024/25,3,0
4,2025-08-16,Tottenham,Burnley,3,0,H,2024/25,3,0


In [182]:
df["home_goal_diff"] = df["home_goals"] - df["away_goals"]
df["away_goal_diff"] = df["away_goals"] - df["home_goals"]

df.head()

,date,home_team,away_team,home_goals,away_goals,result,season,home_points,away_points,home_goal_diff,away_goal_diff
0,2025-08-15,Liverpool,Bournemouth,4,2,H,2024/25,3,0,2,-2
1,2025-08-16,Aston Villa,Newcastle,0,0,D,2024/25,1,1,0,0
2,2025-08-16,Brighton,Fulham,1,1,D,2024/25,1,1,0,0
3,2025-08-16,Sunderland,West Ham,3,0,H,2024/25,3,0,3,-3
4,2025-08-16,Tottenham,Burnley,3,0,H,2024/25,3,0,3,-3


In [183]:
teams = pd.concat([df["home_team"],df["away_team"]]).unique()
teams

array(['Liverpool', 'Aston Villa', 'Brighton', 'Sunderland', 'Tottenham',
       'Wolves', 'Chelsea', "Nott'm Forest", 'Man United', 'Leeds',
       'West Ham', 'Man City', 'Bournemouth', 'Brentford', 'Burnley',
       'Arsenal', 'Crystal Palace', 'Everton', 'Fulham', 'Newcastle'],
      dtype=object)

In [184]:
home_stats = df.groupby("home_team").agg(
played_home=("home_team","count"),
home_wins=("home_points", lambda x: (x==3).sum()),
home_losses=("home_points", lambda x: (x==0).sum()),
home_draws=("home_points", lambda x: (x==1).sum()),
goals_for_home=("home_goals","sum"),
goals_against_home=("away_goals","sum"),
points_home=("home_points","sum")
).reset_index().rename(columns={"home_team":"team"})

away_stats = df.groupby("away_team").agg(
played_away=("away_team","count"),
away_wins=("away_points", lambda x: (x==3).sum()),
away_losses=("away_points", lambda x: (x==0).sum()),
away_draws=("away_points", lambda x: (x==1).sum()),
goals_for_away=("away_goals","sum"),
goals_against_away=("home_goals","sum"),
points_away=("away_points","sum")
).reset_index().rename(columns={"away_team":"team"})

In [185]:
league_table = pd.merge(home_stats,away_stats,on="team")

league_table["played"]=league_table["played_home"]+league_table["played_away"]
league_table["wins"]=league_table["home_wins"]+league_table["away_wins"]
league_table["losses"]=league_table["home_losses"]+league_table["away_losses"]
league_table["draws"]=league_table["home_draws"]+league_table["away_draws"]
league_table["goals_scored"]=league_table["goals_for_home"]+league_table["goals_for_away"]
league_table["goals_conceded"]=league_table["goals_against_home"]+league_table["goals_against_away"]
league_table["goal_difference"]=league_table["goals_scored"]+league_table["goals_conceded"]
league_table["points"]=league_table["points_home"]+league_table["points_away"]

league_table = league_table[["team","played","wins","losses","draws","goals_scored","goals_conceded","goal_difference","points"]]

league_table.head()

,team,played,wins,losses,draws,goals_scored,goals_conceded,goal_difference,points
0,Arsenal,18,13,2,3,33,11,44,42
1,Aston Villa,18,12,3,3,29,19,48,39
2,Bournemouth,18,5,6,7,27,33,60,22
3,Brentford,18,8,8,2,28,26,54,26
4,Brighton,18,6,6,6,26,25,51,24


In [186]:
league_table = league_table.sort_values(
by=["points","goal_difference","goals_scored"],ascending=False
).reset_index(drop=True)

league_table["position"]=league_table.index + 1

league_table = league_table[["position"] + [c for c in league_table.columns if c != "position"]]

league_table.style.hide(axis="index")

position,team,played,wins,losses,draws,goals_scored,goals_conceded,goal_difference,points
1,Arsenal,18,13,2,3,33,11,44,42
2,Man City,18,13,4,1,43,17,60,40
3,Aston Villa,18,12,3,3,29,19,48,39
4,Liverpool,18,10,6,2,30,26,56,32
5,Man United,18,8,5,5,32,28,60,29
6,Chelsea,18,8,5,5,30,19,49,29
7,Sunderland,18,7,4,7,20,18,38,28
8,Brentford,18,8,8,2,28,26,54,26
9,Fulham,18,8,8,2,25,26,51,26
10,Crystal Palace,18,7,6,5,21,20,41,26


In [187]:
home_matches=df[["date","home_team","home_goals","away_goals","home_points"]].copy().rename(columns={"home_team":"team","home_points":"points"})
home_matches["home_away"]="H"

away_matches=df[["date","away_team","away_goals","home_goals","away_points"]].copy().rename(columns={"away_team":"team","away_points":"points"})
away_matches["home_away"]="A"

team_matches = pd.concat([home_matches,away_matches],ignore_index=True)

team_matches = team_matches.sort_values("date").reset_index(drop=True)

team_matches.head()

,date,team,home_goals,away_goals,points,home_away
0,2025-08-15,Liverpool,4,2,3,H
1,2025-08-15,Bournemouth,4,2,0,A
2,2025-08-16,Burnley,3,0,0,A
3,2025-08-16,West Ham,3,0,0,A
4,2025-08-16,Fulham,1,1,1,A


In [188]:
def result_label(points):
    if points==3:
        return "🟢"
    if points==1:
        return "⚪️"
    else:
        return "🔴"
    
team_matches["result"]=team_matches["points"].apply(result_label)

team_matches.head()

,date,team,home_goals,away_goals,points,home_away,result
0,2025-08-15,Liverpool,4,2,3,H,🟢
1,2025-08-15,Bournemouth,4,2,0,A,🔴
2,2025-08-16,Burnley,3,0,0,A,🔴
3,2025-08-16,West Ham,3,0,0,A,🔴
4,2025-08-16,Fulham,1,1,1,A,⚪️


In [189]:
recent_form = team_matches.groupby("team").tail(5).groupby("team")["result"].apply(lambda x: "".join(x))
recent_form

team
Arsenal              🟢🔴🟢🟢🟢
Aston Villa          🟢🟢🟢🟢🟢
Bournemouth       🔴⚪️⚪️⚪️🔴
Brentford           🔴🔴⚪️🟢🟢
Brighton           🔴⚪️🔴⚪️🔴
Burnley            🔴🔴🔴⚪️⚪️
Chelsea            🔴⚪️🟢⚪️🔴
Crystal Palace       🟢🟢🔴🔴🔴
Everton             🟢🟢🔴🔴⚪️
Fulham               🔴🔴🟢🟢🟢
Leeds             🟢⚪️⚪️🟢⚪️
Liverpool          ⚪️⚪️🟢🟢🟢
Man City             🟢🟢🟢🟢🟢
Man United         ⚪️🟢⚪️🔴🟢
Newcastle          ⚪️🟢🔴⚪️🔴
Nott'm Forest        🟢🔴🟢🔴🔴
Sunderland        ⚪️🔴🟢⚪️⚪️
Tottenham           ⚪️🟢🔴🔴🟢
West Ham           ⚪️⚪️🔴🔴🔴
Wolves               🔴🔴🔴🔴🔴
Name: result, dtype: object

In [190]:
league_table = league_table.merge(recent_form,on="team",how="left").rename(columns={"result":"form"})
league_table

,position,team,played,wins,losses,draws,goals_scored,goals_conceded,goal_difference,points,form
0,1,Arsenal,18,13,2,3,33,11,44,42,🟢🔴🟢🟢🟢
1,2,Man City,18,13,4,1,43,17,60,40,🟢🟢🟢🟢🟢
2,3,Aston Villa,18,12,3,3,29,19,48,39,🟢🟢🟢🟢🟢
3,4,Liverpool,18,10,6,2,30,26,56,32,⚪️⚪️🟢🟢🟢
4,5,Man United,18,8,5,5,32,28,60,29,⚪️🟢⚪️🔴🟢
5,6,Chelsea,18,8,5,5,30,19,49,29,🔴⚪️🟢⚪️🔴
6,7,Sunderland,18,7,4,7,20,18,38,28,⚪️🔴🟢⚪️⚪️
7,8,Brentford,18,8,8,2,28,26,54,26,🔴🔴⚪️🟢🟢
8,9,Fulham,18,8,8,2,25,26,51,26,🔴🔴🟢🟢🟢
9,10,Crystal Palace,18,7,6,5,21,20,41,26,🟢🟢🔴🔴🔴


In [191]:
def highlight_position(row):
    pos = row.position
    if pos<=4:
        return ["background-color: lightgreen"] * len(row)
    if pos<=6:
        return ["background-color: khaki"] * len(row)
    if pos>=18:
        return ["background-color: lightcoral"] * len(row)
    else:
        return [""] * len(row)
    
styled_table = league_table.style.apply(highlight_position,axis=1).hide(axis="index")
styled_table

position,team,played,wins,losses,draws,goals_scored,goals_conceded,goal_difference,points,form
1,Arsenal,18,13,2,3,33,11,44,42,🟢🔴🟢🟢🟢
2,Man City,18,13,4,1,43,17,60,40,🟢🟢🟢🟢🟢
3,Aston Villa,18,12,3,3,29,19,48,39,🟢🟢🟢🟢🟢
4,Liverpool,18,10,6,2,30,26,56,32,⚪️⚪️🟢🟢🟢
5,Man United,18,8,5,5,32,28,60,29,⚪️🟢⚪️🔴🟢
6,Chelsea,18,8,5,5,30,19,49,29,🔴⚪️🟢⚪️🔴
7,Sunderland,18,7,4,7,20,18,38,28,⚪️🔴🟢⚪️⚪️
8,Brentford,18,8,8,2,28,26,54,26,🔴🔴⚪️🟢🟢
9,Fulham,18,8,8,2,25,26,51,26,🔴🔴🟢🟢🟢
10,Crystal Palace,18,7,6,5,21,20,41,26,🟢🟢🔴🔴🔴


In [193]:
league_table.to_csv("../data/processed/league_table.csv", index=False)

In [199]:
home_stats["venue"]="home"
away_stats["venue"]="away"

home_stats = home_stats.rename(columns={
    "played_home": "played",
    "home_wins": "wins",
    "home_losses": "losses",
    "home_draws": "draws",
    "goals_for_home": "goals_for",
    "goals_against_home": "goals_against",
    "points_home": "points"
})

away_stats = away_stats.rename(columns={
    "played_away": "played",
    "away_wins": "wins",
    "away_losses": "losses",
    "away_draws": "draws",
    "goals_for_away": "goals_for",
    "goals_against_away": "goals_against",
    "points_away": "points"
})

home_away_stats = pd.concat([home_stats, away_stats], ignore_index=True)

home_away_stats.style.hide(axis="index")

team,played,wins,losses,draws,goals_for,goals_against,points,venue
Arsenal,9,8,0,1,22,4,25,home
Aston Villa,9,7,1,1,15,7,22,home
Bournemouth,9,4,1,4,11,6,16,home
Brentford,9,6,1,2,20,10,20,home
Brighton,9,4,1,4,16,11,16,home
Burnley,9,2,5,2,7,10,8,home
Chelsea,9,4,3,2,13,9,14,home
Crystal Palace,9,2,3,4,9,11,10,home
Everton,9,4,3,2,11,10,14,home
Fulham,9,5,3,1,15,10,16,home


In [200]:
home_away_stats.to_csv("../data/processed/home_away_stats.csv", index=False)